# CAMUS Baseline — Attention Residual U-Net

Thin notebook: all logic lives in the `iteris/` package. This file just installs deps,
loads config, and calls high-level functions.

**Kaggle setup**
1. Add CAMUS dataset to inputs (path used below: `/kaggle/input/datasets/anfaalhossain/camus/CAMUS`)
2. Add `iteris-pkg` Kaggle Dataset to inputs (upload the `iteris/` folder + `configs/` + `requirements.txt`)
3. Run the first cell, **restart the kernel**, then **Run All** from cell 2 onward

**To switch datasets:** change the YAML path in Cell 2. Nothing else changes.

## 0 · Install pinned dependencies
Run this cell **once**, then **Run → Restart Session**, then Run All from cell 2.

In [ ]:
!pip install -r /kaggle/input/iteris-pkg/requirements.txt --quiet --force-reinstall
print('⚠  RESTART KERNEL NOW:  Run → Restart Session, then Run All from cell 2.')

## 1 · Setup — package on path + config

In [ ]:
import sys
sys.path.insert(0, '/kaggle/input/iteris-pkg')

from iteris.config import load_config
from iteris.utils  import get_device

cfg = load_config('/kaggle/input/iteris-pkg/configs/camus.yaml')

# Override Kaggle-specific paths (these change per session / dataset upload)
cfg['data_root']      = '/kaggle/input/datasets/anfaalhossain/camus/CAMUS'
cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Dataset      : {cfg["dataset"]}')
print(f'Image size   : {cfg["image_size"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}')
print(f'Device       : {get_device()}')

## 2 · Train
End-to-end: ingestion → patient-level split → MONAI transforms → cached dataloaders →
Attention Residual U-Net training with cosine schedule + early stopping → best checkpoint saved.

In [ ]:
from iteris.training import run_training

result = run_training(cfg)
model         = result['model']
history       = result['history']
best_dice     = result['best_dice']
test_loader   = result['test_loader']
checkpoint    = result['checkpoint']

print(f'\n✓ Training done. Best val Dice = {best_dice:.4f}  |  Checkpoint: {checkpoint}')

## 3 · Learning curves

In [ ]:
from iteris.visualization import plot_learning_curves
plot_learning_curves(history, cfg, target_dice=0.85)

## 4 · Test-set evaluation
Per-patient Dice + HD95 (pure-torch implementation, no scipy). Writes the CSV that
Week 4 statistical tests and DRL agents consume.

In [ ]:
from iteris.evaluation import evaluate_test_set
scores_df = evaluate_test_set(model, test_loader, cfg)
print(scores_df.head())

## 5 · Export predicted masks
One `.npy` per test sample. These become the DRL environment's `init_mask`.

In [ ]:
from iteris.evaluation import export_predicted_masks
export_predicted_masks(model, test_loader, cfg)

## 6 · Qualitative grid

In [ ]:
from iteris.visualization import plot_qualitative_grid
plot_qualitative_grid(model, test_loader, cfg, n_show=4)

## 7 · Summary JSON
Snapshot consumed by downstream notebooks.

In [ ]:
from iteris.evaluation import save_summary_json
save_summary_json(history, scores_df, cfg, best_dice)